# Interior Product Binder Builder

A point-and-click front-end for the binder generator — no command line needed.

**How to use:**
1. Run the **Setup** cell below once (*Run → Run All*, or Shift+Enter).
2. Fill in your **Project details**.
3. For each product: type a **KEY**, a **Description**, and a **Source**
   (click **Browse…** to pick a local PDF, or paste an `http(s)://` link).
   Open **Optional details** if you need PG / Description 2 / Description 3.
   Click **Add row**. Repeat for every product. Rows that share the same
   Source are grouped onto one page.
4. Click **Generate binder**. The schedule is saved to
   `output/manual_schedule.xlsx` and the PDF to
   `output/interior_product_binder.pdf`.

> Requires `ANTHROPIC_API_KEY` in a `.env` file (see the README).


In [ ]:
# --- Setup: run this cell once ---
import os

# Make sure we're at the repo root so `src`, `output/`, and `.env` resolve.
if not os.path.exists("main.py") and os.path.exists(os.path.join("..", "main.py")):
    os.chdir("..")

from dotenv import load_dotenv
load_dotenv()

from src import excel_reader, pipeline

print("Working directory:", os.getcwd())
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("  -> Copy .env.example to .env and add your key before generating.")


In [ ]:
# --- Product entry form ---
import os
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

rows = []  # collected product rows
FULL = widgets.Layout(width="90%")

# --- Project details ---
project_name = widgets.Text(value="Untitled Project", description="Project:",
                            layout=widgets.Layout(width="70%"))
prepared_by = widgets.Text(value=os.environ.get("COMPANY_NAME", ""),
                           description="Prepared by:",
                           layout=widgets.Layout(width="70%"))

# --- Required per-product fields ---
key_in = widgets.Text(description="KEY", placeholder="e.g. U-36")
desc_in = widgets.Text(description="Description",
                       placeholder="e.g. VANITY CABINET - 60-IN", layout=FULL)
source_in = widgets.Text(description="Source",
                         placeholder="local PDF path or URL (blank = owner input needed)",
                         layout=widgets.Layout(width="72%"))
browse_btn = widgets.Button(description="Browse...", icon="folder-open",
                            layout=widgets.Layout(width="16%"))

# --- Optional per-product fields (collapsed by default) ---
pg_in = widgets.Text(description="Page (PG)", placeholder="optional")
desc2_in = widgets.Text(description="Description 2", placeholder="optional", layout=FULL)
desc3_in = widgets.Text(description="Description 3", placeholder="optional", layout=FULL)
optional_acc = widgets.Accordion(children=[widgets.VBox([pg_in, desc2_in, desc3_in])])
optional_acc.set_title(0, "Optional details (PG, Description 2, Description 3)")
optional_acc.selected_index = None  # start collapsed

add_btn = widgets.Button(description="Add row", button_style="info", icon="plus")
clear_btn = widgets.Button(description="Clear all rows", button_style="warning")
gen_btn = widgets.Button(description="Generate binder", button_style="success", icon="check")

table_out = widgets.Output()
status_out = widgets.Output()

FIELDS = [key_in, desc_in, source_in, pg_in, desc2_in, desc3_in]


def on_browse(_):
    """Open the native file explorer and put the chosen path in the Source box."""
    with status_out:
        clear_output()
        try:
            import tkinter as tk
            from tkinter import filedialog
            root = tk.Tk()
            root.withdraw()
            root.attributes("-topmost", True)
            path = filedialog.askopenfilename(
                title="Select a product spec sheet (PDF)",
                filetypes=[("PDF files", "*.pdf"), ("All files", "*.*")])
            root.destroy()
            if path:
                source_in.value = path
        except Exception as e:
            print(f"File picker unavailable ({e}). Please paste the path manually.")


def render_table():
    with table_out:
        clear_output()
        if not rows:
            display(HTML("<i>No rows yet - add a product above.</i>"))
            return
        html = ("<table style='border-collapse:collapse' border='1' cellpadding='4'>"
                "<tr><th>#</th><th>KEY</th><th>DESCRIPTION</th><th>PG</th>"
                "<th>SOURCE</th></tr>")
        for i, r in enumerate(rows, 1):
            src = r['PATH'] or "<i>(owner input needed)</i>"
            html += (f"<tr><td>{i}</td><td>{r['KEY']}</td><td>{r['DESCRIPTION']}</td>"
                     f"<td>{r['PG']}</td><td>{src}</td></tr>")
        html += "</table>"
        display(HTML(html))


def on_add(_):
    with status_out:
        clear_output()
        if not key_in.value.strip():
            print("KEY is required.")
            return
    rows.append({
        "KEY": key_in.value.strip(),
        "DESCRIPTION": desc_in.value.strip(),
        "PG": pg_in.value.strip(),
        "PATH": source_in.value.strip(),
        "DESCRIPTION2": desc2_in.value.strip(),
        "DESCRIPTION3": desc3_in.value.strip(),
    })
    for w in FIELDS:
        w.value = ""
    render_table()


def on_clear(_):
    rows.clear()
    render_table()
    with status_out:
        clear_output()


def on_generate(_):
    with status_out:
        clear_output()
        if not rows:
            print("Add at least one product row first.")
            return
        if not os.environ.get("ANTHROPIC_API_KEY"):
            print("ANTHROPIC_API_KEY is not set - add it to .env and re-run Setup.")
            return
        os.makedirs("output", exist_ok=True)
        base = pipeline.unique_output_base(
            project_name.value or "Untitled Project", prepared_by.value)
        xlsx_path = base + ".xlsx"
        pdf_path = base + ".pdf"
        excel_reader.save_schedule(rows, xlsx_path)
        print(f"Saved schedule: {xlsx_path}\n")
        results = pipeline.generate_from_xlsx(
            xlsx_path, pdf_path,
            project_name.value or "Untitled Project",
            prepared_by.value, log=print)
        ok = sum(1 for r in results if not r.error)
        print(f"\nDone: {ok}/{len(results)} product group(s) succeeded.")
        print(f"Binder PDF: {pdf_path}")


browse_btn.on_click(on_browse)
add_btn.on_click(on_add)
clear_btn.on_click(on_clear)
gen_btn.on_click(on_generate)

display(widgets.HTML("<h3>1. Project details</h3>"))
display(project_name, prepared_by)
display(widgets.HTML("<h3>2. Add products</h3>"))
display(widgets.HBox([key_in, desc_in]))
display(widgets.HBox([source_in, browse_btn]))
display(optional_acc)
display(widgets.HBox([add_btn, clear_btn]))
display(table_out)
display(widgets.HTML("<h3>3. Generate</h3>"))
display(gen_btn)
display(status_out)
render_table()
